In [ ]:
!pip install openai

In [ ]:
!pip install sentence-transformers

In [ ]:
import os
import time
import json
import re
import ast
from jsonschema import validate as jsonschema_validate, ValidationError
import subprocess
from openai import OpenAI
from datetime import datetime

In [ ]:
# --- CONFIGURATION --- #
base_dir = os.path.abspath(os.path.join(os.path.dirname("."), ".."))
TEST_NAME = "blood_donation_scenario"
MODEL_NAME = "gpt-4.1"
REQUIREMENT_BASE_ID = "r17"
TEMPERATURE = 0

# Load compliance requirement schema from file
schema_path = os.path.join(base_dir, "data", "input", "formats", "compliance_requirements_format.json")
with open(schema_path, "r", encoding="utf-8") as schema_file:
    COMPLIANCE_REQUIREMENT_SCHEMA = json.load(schema_file)

# Load API key
api_keys_path = os.path.join(base_dir, "config", "api_keys.json")
api_keys = json.load(open(api_keys_path, "r", encoding="utf-8"))
client = OpenAI(api_key=api_keys["api_openai"])

# === UTILITY FUNCTIONS ===
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_prompt(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def normalize_id(raw_id):
    return re.sub(r"[’'´`]+$", "", str(raw_id).strip())

def safe_json_load(text):
    text = text.strip().strip("`").strip()
    if text.lower().startswith("json"):
        text = text[4:].strip()
    try:
        return json.loads(text)
    except Exception:
        try:
            return ast.literal_eval(text)
        except Exception:
            return None

def log_error(path, msg):
    with open(path, "a", encoding="utf-8") as err_file:
        err_file.write(msg + "\n\n")

def save_metadata(meta_info, meta_path):
    with open(meta_path, "w", encoding="utf-8") as meta_file:
        json.dump(meta_info, meta_file, indent=4)

def build_metadata(model, provider, prompt_file, temperature, input_file, output_file, error_log,
                   extra_info=None, process_model_pdf=None, start_time=None, end_time=None):
    execution_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    execution_time = round(end_time - start_time, 2) if start_time and end_time else None
    meta = {
        "model": model,
        "provider": provider,
        "prompt_file": prompt_file,
        "temperature": temperature,
        "execution_timestamp": execution_timestamp,
        "execution_time_seconds": execution_time,
        "input_file": input_file,
        "output_file": output_file,
        "error_log": error_log
    }
    if process_model_pdf is not None:
        meta["process_model_pdf"] = process_model_pdf
    if extra_info:
        meta.update(extra_info)
    return meta


def query_llm(model_name, system_prompt, user_input, temperature, max_tokens=4096):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input if isinstance(user_input, str) else json.dumps(user_input, ensure_ascii=False)}
    ]
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=1
    )
    return response.choices[0].message.content.strip()

def validate_formalization_format(data):
    try:
        jsonschema_validate(instance=data, schema=COMPLIANCE_REQUIREMENT_SCHEMA)
        return True, "Valid"
    except ValidationError as ve:
        return False, f"Schema validation error: {ve.message}"

def validate_deviation_format(data):
    if not isinstance(data, (list, dict)):
        return False, "Expected a list or dict as output"
    return True, "Format OK"

def filter_requirements(requirements, requirement_base_id):
    return [
        req for req in requirements
        if normalize_id(req.get("ID", "")) == requirement_base_id
    ]

def ensure_list_fields(response_json):
    def fix_action(action):
        if "data_values" in action:
            for dv in action["data_values"]:
                if "domain" in dv and not isinstance(dv["domain"], list):
                    dv["domain"] = [dv["domain"]]
                if "value" in dv and not isinstance(dv["value"], list):
                    dv["value"] = [dv["value"]]
    if "precondition" in response_json:
        for clause in ["and", "or", "not"]:
            for action in response_json["precondition"].get(clause, []):
                fix_action(action)
    if "norms" in response_json:
        for norm in response_json["norms"]:
            fix_action(norm["action"])
    return response_json

def process_formalization(requirements, base_prompt, output_dir, model_name, temperature, requirement_base_id):
    merged_data = []
    error_log_path = os.path.join(output_dir, "error_log.txt")
    filtered_requirements = filter_requirements(requirements, requirement_base_id)
    version_map = {str(req.get("version", "1")): req for req in filtered_requirements}
    text_v1 = version_map.get("1", {}).get("text", "").strip()
    text_v2 = version_map.get("2", {}).get("text", "").strip()

    versions_to_formalize = []
    if text_v1 and text_v2:
        versions_to_formalize = ["1", "2"]
    elif text_v1:
        versions_to_formalize = ["1"]
    elif text_v2:
        versions_to_formalize = ["2"]
    else:
        print(f"WARNING: Both versions of {requirement_base_id} are empty. Skipping.")
        return []

    for version in versions_to_formalize:
        req = version_map[version]
        user_input = json.dumps({
            "ID": req.get("ID"),
            "version": version,
            "text": req.get("text", "")
        }, ensure_ascii=False)
        response_text = None
        try:
            response_text = query_llm(model_name, base_prompt, user_input, temperature)
            response_json = safe_json_load(response_text)
            response_json = ensure_list_fields(response_json)
            is_valid, validation_msg = validate_formalization_format(response_json)
            if is_valid:
                response_json["id"] = f"{requirement_base_id}_{version}"
                response_json["input_id"] = req.get("ID")
                response_json["input_version"] = version
                merged_data.append(response_json)
                print(f"COMPLETED: Processed {req.get('ID')} v{version}")
            else:
                raise ValueError(validation_msg)
        except Exception as e:
            raw_output_path = os.path.join(output_dir, f"raw_response_{req.get('ID')}_v{version}.txt")
            save_json({"response": response_text or "No response (exception before LLM call)"}, raw_output_path)
            log_error(error_log_path, f"Error for ID {req.get('ID')} (v{version}): {str(e)}")
            print(f"ERROR: Error for ID {req.get('ID')} v{version} (logged)")
    return merged_data

# === EXECUTION SETUP ===
formalize_prompt_path = os.path.join(base_dir, "data", "input", "prompts", "formalize_requirements_prompt.txt")
requirements_path = os.path.join(base_dir, "data", "input", "requirements", TEST_NAME, TEST_NAME + ".json")
output_dir_formalized = os.path.join(base_dir, "data", "output", "01_formalized_requirements", TEST_NAME)
output_dir_deviations = os.path.join(base_dir, "data", "output", "03_compliance_deviations", TEST_NAME)
os.makedirs(output_dir_formalized, exist_ok=True)
os.makedirs(output_dir_deviations, exist_ok=True)

# Load data
requirements = load_json(requirements_path)
formalize_prompt = load_prompt(formalize_prompt_path)

# Run
formalized_output = process_formalization(requirements, formalize_prompt, output_dir_formalized, MODEL_NAME, TEMPERATURE, REQUIREMENT_BASE_ID)


In [ ]:
# STEP 1 - FORMALIZE COMPLIANCE REQUIREMENTS
start_time = time.time()
base_prompt = load_prompt(formalize_prompt_path)

formalized = process_formalization(
    requirements=requirements,
    base_prompt=base_prompt,
    output_dir=output_dir_formalized,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    requirement_base_id=REQUIREMENT_BASE_ID
)

if formalized:
    out_path = os.path.join(output_dir_formalized, f"formalized_requirements_{REQUIREMENT_BASE_ID}.json")
    save_json(formalized, out_path)
    print("COMPLETED: Saved:", out_path)
else:
    out_path = None

end_time = time.time()

formalization_meta = build_metadata(
    model=MODEL_NAME,
    provider="openai",
    prompt_file=os.path.basename(formalize_prompt_path),
    temperature=TEMPERATURE,
    input_file=os.path.basename(requirements_path),
    output_file=os.path.basename(out_path) if out_path else None,
    error_log="error_log.txt",
    start_time=start_time,
    end_time=end_time
)

meta_path = os.path.join(output_dir_formalized, f"execution_formalizing_metadata_{REQUIREMENT_BASE_ID}.json")
save_metadata(formalization_meta, meta_path)
print("COMPLETED: Formalization metadata saved:", meta_path)


In [ ]:
# STEP 2 - EXTRACT ATOMIC CHANGE OPERATIONS (DELTA)

# Setup paths
delta_output_path = os.path.join(base_dir, "data", "output", "02_delta", TEST_NAME, f"delta_{REQUIREMENT_BASE_ID}.json")
delta_output_dir = os.path.dirname(delta_output_path)
formalized_path = os.path.join(base_dir, "data", "output", "01_formalized_requirements", TEST_NAME, f"formalized_requirements_{REQUIREMENT_BASE_ID}.json")
calculate_delta_script = os.path.join(base_dir, "scripts", "calculate_atomic_delta.py")
os.makedirs(delta_output_dir, exist_ok=True)

# Start timer
start_time = time.time()

# Load formalized data
formalized = load_json(formalized_path)

# Extract versions
req_v1 = next((r for r in formalized if str(r.get("input_version")) == "1"), None)
req_v2 = next((r for r in formalized if str(r.get("input_version")) == "2"), None)

changes = []

if req_v1 is None and req_v2 is not None:
    changes.append([
        f"{REQUIREMENT_BASE_ID}_c1",
        "compliance",
        { "from": None, "to": req_v2 },
        "added"
    ])
elif req_v1 is not None and req_v2 is None:
    changes.append([
        f"{REQUIREMENT_BASE_ID}_c1",
        "compliance",
        { "from": req_v1, "to": None },
        "deleted"
    ])

# Either save simple delta or call external script
if changes:
    delta = {
        "requirement_id": REQUIREMENT_BASE_ID,
        "from_version": 1,
        "to_version": 2,
        "changes": changes
    }
    save_json(delta, delta_output_path)
    print("COMPLETED: Delta file saved directly:", delta_output_path)
else:
    # Run delta calculation script
    command = [
        "python", calculate_delta_script,
        "--input", formalized_path,
        "--output", delta_output_path
    ]
    subprocess.run(command, check=True)
    print("COMPLETED: Extraction of delta completed via script.")

# End timer
end_time = time.time()

# Save metadata
delta_meta = build_metadata(
    model="N/A",
    provider="internal_script",
    prompt_file=os.path.basename(calculate_delta_script),
    temperature="N/A",
    input_file=os.path.basename(formalized_path),
    output_file=os.path.basename(delta_output_path),
    error_log="error_log.txt",
    start_time=start_time,
    end_time=end_time
)

meta_path = os.path.join(delta_output_dir, f"execution_delta_metadata_{REQUIREMENT_BASE_ID}.json")
save_metadata(delta_meta, meta_path)
print("COMPLETED: Delta metadata saved:", meta_path)

In [ ]:
# STEP 3 - DETECT COMPLIANCE DEVIATIONS
from datetime import datetime

# === LOAD API KEYS AND INITIALIZE CLIENT ===
base_dir = os.path.abspath(os.path.join(os.path.dirname("."), ".."))
api_keys_path = os.path.join(base_dir, "config", "api_keys.json")
api_keys = json.load(open(api_keys_path, "r", encoding="utf-8"))
client = OpenAI(api_key=api_keys["api_openai"])

# === LOAD REQUIREMENTS AND FORMALIZED DATA ===
with open(requirements_path, "r", encoding="utf-8") as f:
    all_requirements = json.load(f)

natural_language_versions = [r["text"] for r in all_requirements if r["ID"] == REQUIREMENT_BASE_ID]
if len(natural_language_versions) != 2:
    raise ValueError(f"Expected exactly 2 versions for requirement ID {REQUIREMENT_BASE_ID}, found {len(natural_language_versions)}")

with open(os.path.join(output_dir_formalized, f"formalized_requirements_{REQUIREMENT_BASE_ID}.json"), "r", encoding="utf-8") as f:
    formalized_json = json.load(f)

with open(delta_output_path, "r", encoding="utf-8") as f:
    delta_json = json.load(f)

# 🔧 Fix for single-entry formalized JSON
if isinstance(formalized_json, dict):
    formalized_json = [formalized_json]

req_v1 = next((r for r in formalized_json if str(r.get("input_version")) == "1"), None)
req_v2 = next((r for r in formalized_json if str(r.get("input_version")) == "2"), None)

formalized_v1_text = f"Formalized version 1:\n{json.dumps(req_v1, indent=2, ensure_ascii=False)}\n\n" if req_v1 else ""
formalized_v2_text = f"Formalized version 2:\n{json.dumps(req_v2, indent=2, ensure_ascii=False)}\n\n" if req_v2 else ""

# === LOAD AND UPLOAD PROCESS MODEL PDF ===
process_model_pdf_path = os.path.join(base_dir, "data", "input", "process_models", TEST_NAME, TEST_NAME + ".pdf")

uploaded_pdf = client.files.create(
    file=open(process_model_pdf_path, "rb"),
    purpose="user_data"
)

# === PROMPT ===
prompt = (
    "You are a compliance analyst.\n\n"
    "You are given two versions of a compliance requirement (both in natural language and formalized format), "
    "and the extracted delta (atomic changes between both versions).\n\n"
    "Assume the following:\n"
    "- The process model represents the actual behavior of the system.\n"
    "- The process model has not changed between Version 1 and Version 2.\n"
    "- Therefore, any new constraints introduced in Version 2 are not reflected in the process model unless explicitly stated.\n"
    "- This means that if the requirement becomes stricter, the process MIGHT violate it — this is a case of 'non-compliance'.\n"
    "- Do not use conditional expressions such as 'if the process still...', 'may violate', 'could cause', or 'might result in'.\n"
    "- Use deterministic and declarative phrasing: state the violation directly when applicable.\n\n"
    f"Version 1 (natural language):\n{natural_language_versions[0]}\n\n"
    f"Version 2 (natural language):\n{natural_language_versions[1]}\n\n"
    f"{formalized_v1_text}"
    f"{formalized_v2_text}"
    f"Delta between versions:\n{json.dumps(delta_json, indent=2, ensure_ascii=False)}\n\n"
    "You will also receive the process model as a PDF file. Use it to identify compliance deviations.\n\n"
    "For each atomic change, identify all potential compliance deviations that could arise due to this change.\n"
    "Also indicate the most likely type of compliance deviation it could cause, choosing from:\n"
    "- 'non-compliance': the process violates a stricter requirement introduced in Version 2\n"
    "- 'over-compliance': the process unnecessarily restricts behavior or continues enforcing outdated constraints\n"
    "- 'no impact': the change does not affect compliance\n"
    "- 'ambiguous': impact cannot be determined without additional technical or domain knowledge (avoid this when a stricter requirement is clearly introduced)\n\n"
    "Use the following JSON format:\n"
    "[\n"
    "  {\n"
    "    \"type\": \"non-compliance | over-compliance | no impact | ambiguous\",\n"
    "    \"reason\": \"concise explanation of the compliance deviation, using direct and assertive language\",\n"
    "    \"impacted_element\": \"process elementS that ARE affected (e.g., 'issue SIM card')\"\n"
    "  }\n"
    "]"
)

start_time = time.time()
# === CALL LLM AND PROCESS DEVIATIONS ===
os.makedirs(output_dir_deviations, exist_ok=True)

try:
    print("Calling LLM to detect compliance deviations...")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an expert in compliance deviations."},
            {
                "role": "user",
                "content": [
                    {"type": "file", "file": { "file_id": uploaded_pdf.id }},
                    {"type": "text", "text": prompt}
                ]

            }
        ]
    )

    response_text = response.choices[0].message.content.strip()

    try:
        raw_deviations = safe_json_load(response_text)
        if not isinstance(raw_deviations, list):
            print("ERROR: LLM output is not a list. Raw response:")
            print(response_text)
        else:
            structured_output = {
                "requirement_id": REQUIREMENT_BASE_ID,
                "from_version": 1,
                "to_version": 2,
                "deviations": []
            }

            # Extract change_ids from delta
            if isinstance(delta_json, dict) and isinstance(delta_json.get("changes"), list):
                change_ids = [c[0] if isinstance(c, list) else c.get("change_id") for c in delta_json["changes"]]
            elif isinstance(delta_json, list):
                change_ids = [c[0] if isinstance(c, list) else c.get("change_id") for c in delta_json]
            else:
                raise ValueError("Unexpected delta_json format")

            for idx, d in enumerate(raw_deviations):
                structured_output["deviations"].append({
                    "deviation_id": f"{REQUIREMENT_BASE_ID}_d{idx+1}",
                    "deviation_type": d.get("type"),
                    "description": d.get("reason"),
                    "process_model_reference": [d.get("impacted_element")] if d.get("impacted_element") else [],
                    "change_id": change_ids[idx] if idx < len(change_ids) else f"{REQUIREMENT_BASE_ID}_change_{idx+1}"
                })

            # === SAVE TO FILE ===
            final_output = [structured_output]
            deviation_output_path = os.path.join(output_dir_deviations, f"deviations_{REQUIREMENT_BASE_ID}.json")

            with open(deviation_output_path, "w", encoding="utf-8") as f:
                json.dump(final_output, f, indent=2, ensure_ascii=False)

            print(f"COMPLETED: Deviations saved to: {deviation_output_path}")

    except json.JSONDecodeError:
        print("ERROR: Failed to parse LLM output as JSON. Raw response:")
        print(response_text)

except Exception as e:
    print("ERROR: Exception during LLM call:", str(e))

end_time = time.time()

# === Save Metadata ===
deviation_meta = build_metadata(
    model=MODEL_NAME,
    provider="openai",
    prompt_file="inline_prompt_with_pdf",  # or name your prompt file if it’s from disk
    temperature=0,
    input_file=os.path.basename(requirements_path),
    output_file=os.path.basename(deviation_output_path),
    error_log="error_log.txt",
    process_model_pdf=os.path.basename(process_model_pdf_path),
    start_time=start_time,
    end_time=end_time
)

meta_path = os.path.join(output_dir_deviations, f"execution_deviation_metadata_{REQUIREMENT_BASE_ID}.json")
save_metadata(deviation_meta, meta_path)
print("COMPLETED: Deviation metadata saved:", meta_path)
print("COMPLETED: Compliance deviation analysis completed.")

In [ ]:
# Ablation analysis step 3
# STEP 3 - DETECT COMPLIANCE DEVIATIONS
from datetime import datetime
import os
import json
import time
from openai import OpenAI

# === LOAD API KEYS AND INITIALIZE CLIENT ===
base_dir = os.path.abspath(os.path.join(os.path.dirname("."), ".."))
api_keys_path = os.path.join(base_dir, "config", "api_keys.json")
api_keys = json.load(open(api_keys_path, "r", encoding="utf-8"))
client = OpenAI(api_key=api_keys["api_openai"])

# === LOAD REQUIREMENTS AND FORMALIZED DATA ===
with open(requirements_path, "r", encoding="utf-8") as f:
    all_requirements = json.load(f)

natural_language_versions = [
    r["text"] for r in all_requirements if r["ID"] == REQUIREMENT_BASE_ID
]
if len(natural_language_versions) != 2:
    raise ValueError(
        f"Expected exactly 2 versions for requirement ID {REQUIREMENT_BASE_ID}, "
        f"found {len(natural_language_versions)}"
    )

with open(
    os.path.join(output_dir_formalized, f"formalized_requirements_{REQUIREMENT_BASE_ID}.json"),
    "r",
    encoding="utf-8"
) as f:
    formalized_json = json.load(f)

with open(delta_output_path, "r", encoding="utf-8") as f:
    delta_json = json.load(f)

# 🔧 Fix for single-entry formalized JSON
if isinstance(formalized_json, dict):
    formalized_json = [formalized_json]

req_v1 = next((r for r in formalized_json if str(r.get("input_version")) == "1"), None)
req_v2 = next((r for r in formalized_json if str(r.get("input_version")) == "2"), None)

formalized_v1_text = (
    f"Formalized version 1:\n{json.dumps(req_v1, indent=2, ensure_ascii=False)}\n\n"
    if req_v1 else ""
)
formalized_v2_text = (
    f"Formalized version 2:\n{json.dumps(req_v2, indent=2, ensure_ascii=False)}\n\n"
    if req_v2 else ""
)

# === LOAD BPMN AS TEXT (instead of uploading a file) ===
process_model_bpmn_path = os.path.join(
    base_dir, "data", "input", "process_models", TEST_NAME, TEST_NAME + ".bpmn"
)

with open(process_model_bpmn_path, "r", encoding="utf-8") as f:
    bpmn_xml = f.read()

# === PROMPT ===
prompt = (
    "You are a compliance analyst.\n\n"
    "You are given two versions of a compliance requirement (both in natural language and "
    "formalized format), and the extracted delta (atomic changes between both versions).\n\n"
    "Assume the following:\n"
    "- The process model represents the actual behavior of the system.\n"
    "- The process model has not changed between Version 1 and Version 2.\n"
    "- Therefore, any new constraints introduced in Version 2 are not reflected in the process model unless explicitly stated.\n"
    "- This means that if the requirement becomes stricter, the process violates it — this is a case of 'non-compliance'.\n"
    "- Do not use conditional expressions such as 'if the process still...', 'may violate', 'could cause', or 'might result in'.\n"
    "- Use deterministic and assertive phrasing.\n\n"
    f"Version 1 (natural language):\n{natural_language_versions[0]}\n\n"
    f"Version 2 (natural language):\n{natural_language_versions[1]}\n\n"
    f"{formalized_v1_text}"
    f"{formalized_v2_text}"
    f"Delta between versions:\n{json.dumps(delta_json, indent=2, ensure_ascii=False)}\n\n"
    "The following is the BPMN process model (XML content):\n\n"
    "----- BEGIN BPMN XML -----\n"
    f"{bpmn_xml}\n"
    "----- END BPMN XML -----\n\n"
    "For each atomic change, identify all resulting compliance deviations.\n"
    "Deviation types allowed:\n"
    "- 'non-compliance'\n"
    "- 'over-compliance'\n"
    "- 'no impact'\n"
    "- 'ambiguous'\n\n"
    "Output JSON strictly in this format:\n"
    "[\n"
    "  {\n"
    "    \"type\": \"non-compliance | over-compliance | no impact | ambiguous\",\n"
    "    \"reason\": \"concise explanation using deterministic language\",\n"
    "    \"impacted_element\": \"name of BPMN element(s) affected\"\n"
    "  }\n"
    "]"
)

start_time = time.time()

# === CALL LLM AND PROCESS DEVIATIONS ===
os.makedirs(output_dir_deviations, exist_ok=True)

try:
    print("Calling LLM to detect compliance deviations...")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an expert in compliance deviations."},
            {"role": "user", "content": prompt}
        ]
    )

    response_text = response.choices[0].message.content.strip()

    try:
        raw_deviations = safe_json_load(response_text)
        if not isinstance(raw_deviations, list):
            print("ERROR: LLM output is not a list. Raw response:")
            print(response_text)
        else:
            structured_output = {
                "requirement_id": REQUIREMENT_BASE_ID,
                "from_version": 1,
                "to_version": 2,
                "deviations": []
            }

            # Extract change_ids from delta
            if isinstance(delta_json, dict) and isinstance(delta_json.get("changes"), list):
                change_ids = [
                    c[0] if isinstance(c, list) else c.get("change_id")
                    for c in delta_json["changes"]
                ]
            elif isinstance(delta_json, list):
                change_ids = [
                    c[0] if isinstance(c, list) else c.get("change_id")
                    for c in delta_json
                ]
            else:
                raise ValueError("Unexpected delta_json format")

            for idx, d in enumerate(raw_deviations):
                structured_output["deviations"].append({
                    "deviation_id": f"{REQUIREMENT_BASE_ID}_d{idx+1}",
                    "deviation_type": d.get("type"),
                    "description": d.get("reason"),
                    "process_model_reference": (
                        [d.get("impacted_element")] if d.get("impacted_element") else []
                    ),
                    "change_id": (
                        change_ids[idx] if idx < len(change_ids)
                        else f"{REQUIREMENT_BASE_ID}_change_{idx+1}"
                    )
                })

            # === SAVE TO FILE ===
            final_output = [structured_output]
            deviation_output_path = os.path.join(
                output_dir_deviations,
                f"deviations_{REQUIREMENT_BASE_ID}.json"
            )

            with open(deviation_output_path, "w", encoding="utf-8") as f:
                json.dump(final_output, f, indent=2, ensure_ascii=False)

            print(f"COMPLETED: Deviations saved to: {deviation_output_path}")

    except json.JSONDecodeError:
        print("ERROR: Failed to parse LLM output as JSON. Raw response:")
        print(response_text)

except Exception as e:
    print("ERROR: Exception during LLM call:", str(e))

end_time = time.time()

# === Save Metadata ===
deviation_meta = build_metadata(
    model=MODEL_NAME,
    provider="openai",
    prompt_file="inline_prompt_with_bpmn",
    temperature=0,
    input_file=os.path.basename(requirements_path),
    output_file=os.path.basename(deviation_output_path),
    error_log="error_log.txt",
    process_model_bpmn=os.path.basename(process_model_bpmn_path),
    start_time=start_time,
    end_time=end_time
)

meta_path = os.path.join(
    output_dir_deviations,
    f"execution_deviation_metadata_{REQUIREMENT_BASE_ID}.json"
)
save_metadata(deviation_meta, meta_path)
print("COMPLETED: Deviation metadata saved:", meta_path)
print("COMPLETED: Compliance deviation analysis completed.")